# Day 8 — Repeat-Purchase Prediction

**Phase:** plan's Day 9–10 churn model, reframed onto a target that actually exists in Olist.

### Why "repeat", not "churn"
Churn needs a recurring relationship to defect from. Olist is 97% one-time buyers, so there is no churn
label. The honest, trainable target is **binary repeat: did this customer ever place a 2nd order.**

### The leakage rule (the whole reason this notebook is trustworthy)
Features are built from the customer's **first order only**. Aggregating across all orders would leak the
2nd purchase into the features of repeat customers. We predict repeat from what was knowable the moment the
first experience ended. `segment` and any all-order average are excluded for this reason. First-order
`review_score` is allowed: it happens before the repeat decision.

### Metrics
3% positive class. Accuracy is useless here (predict "never" and score 97%). We report **ROC-AUC, PR-AUC,
and precision/recall on the positive class.** Expect a modest AUC; a weak result is itself a finding
(repeat is largely structural, not explained by first-order experience).


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 140)

CWD = Path.cwd()
ROOT = CWD.parent if CWD.name == 'notebooks' else CWD
assert (ROOT / 'data' / 'processed').exists(), f"data/processed not found from ROOT={ROOT}"
PROC = ROOT / 'data' / 'processed'
MODELS = ROOT / 'models'; MODELS.mkdir(exist_ok=True)

master = pd.read_parquet(PROC / 'olist_master.parquet')
master['order_purchase_timestamp'] = pd.to_datetime(master['order_purchase_timestamp'])
print('master:', master.shape)

master: (99441, 31)


## 1. First-order table + repeat target

`idxmin` on the purchase timestamp grabs each customer's earliest order **as one intact row**. This is
deliberate: using `.groupby().first()` would pull the first *non-null* value per column and silently mix
columns from different orders, which would leak. We take the whole earliest-order row, NaNs and all.

In [2]:
# index of each customer's earliest order
first_idx = master.groupby('customer_unique_id')['order_purchase_timestamp'].idxmin()
first = master.loc[first_idx].copy()

# target: did the customer ever order more than once
order_counts = master.groupby('customer_unique_id')['order_id'].nunique()
first['repeat'] = (first['customer_unique_id'].map(order_counts) > 1).astype(int)

base_rate = first['repeat'].mean()
print(f"customers: {len(first):,}")
print(f"repeat (positive) rate: {base_rate*100:.2f}%  ({first['repeat'].sum():,} positives)")

customers: 96,096
repeat (positive) rate: 3.12%  (2,997 positives)


## 2. Feature set (first order only)

Numeric: delivery experience, order value, basket size, freight, first-order review, installments.
Categorical: payment type, customer state. Nothing aggregated across orders, nothing post-repeat.

In [3]:
numeric_feats = ['delivery_time_days', 'days_vs_estimate', 'is_late',
                 'total_payment', 'item_count', 'total_freight',
                 'distinct_products', 'review_score', 'payment_count']
cat_feats = ['main_payment_type', 'customer_state']

# is_late may be bool/object with NaN -> make it numeric, keep NaN for the imputer
first['is_late'] = pd.to_numeric(first['is_late'], errors='coerce')

X = first[numeric_feats + cat_feats].copy()
y = first['repeat'].values
print('feature matrix:', X.shape)
print('null counts:')
print(X[numeric_feats].isna().sum())

feature matrix: (96096, 11)
null counts:
delivery_time_days    2845
days_vs_estimate      2845
is_late                  0
total_payment            1
item_count             707
total_freight          707
distinct_products      707
review_score           994
payment_count            1
dtype: int64


## 3. Train / test split + preprocessing

Stratified split keeps the 3% positives balanced across train and test. Preprocessing lives inside a
ColumnTransformer so imputation/scaling/encoding fit on train only.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

num_pipe = Pipeline([('imp', SimpleImputer(strategy='median')),
                     ('sc', StandardScaler())])
cat_pipe = Pipeline([('imp', SimpleImputer(strategy='constant', fill_value='missing')),
                     ('oh', OneHotEncoder(handle_unknown='ignore'))])

pre = ColumnTransformer([('num', num_pipe, numeric_feats),
                         ('cat', cat_pipe, cat_feats)])
print('train:', X_train.shape, '| test:', X_test.shape)
print('train positives:', int(y_train.sum()), '| test positives:', int(y_test.sum()))

train: (76876, 11) | test: (19220, 11)
train positives: 2398 | test positives: 599


## 4. Evaluation helper

One function so both models are judged the same way, on the metrics that matter for a 3% class.

In [7]:
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             classification_report, confusion_matrix)

def evaluate(name, model, X_te, y_te):
    proba = model.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(f"===== {name} =====")
    print(f"ROC-AUC : {roc_auc_score(y_te, proba):.3f}")
    print(f"PR-AUC  : {average_precision_score(y_te, proba):.3f}  (base rate {y_te.mean():.3f})")
    print("confusion matrix [tn fp / fn tp]:")
    print(confusion_matrix(y_te, pred))
    print(classification_report(y_te, pred, digits=3, zero_division=0))
    return proba

## 5. Logistic regression baseline
`class_weight='balanced'` so the 3% positives aren't ignored.

In [8]:
from sklearn.linear_model import LogisticRegression

lr = Pipeline([('pre', pre),
               ('clf', LogisticRegression(max_iter=1000, class_weight='balanced'))])
lr.fit(X_train, y_train)
lr_proba = evaluate('Logistic Regression', lr, X_test, y_test)

===== Logistic Regression =====
ROC-AUC : 0.582
PR-AUC  : 0.042  (base rate 0.031)
confusion matrix [tn fp / fn tp]:
[[10245  8376]
 [  257   342]]
              precision    recall  f1-score   support

           0      0.976     0.550     0.704     18621
           1      0.039     0.571     0.073       599

    accuracy                          0.551     19220
   macro avg      0.507     0.561     0.388     19220
weighted avg      0.946     0.551     0.684     19220



## 6. XGBoost
`scale_pos_weight = neg/pos` (~31) to counter the imbalance. Falls back to sklearn
HistGradientBoosting if xgboost isn't importable.

In [9]:
spw = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"scale_pos_weight = {spw:.1f}")

try:
    from xgboost import XGBClassifier
    booster = XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric='aucpr',
        random_state=42, n_jobs=-1)
    model_name = 'XGBoost'
except Exception as e:
    print('xgboost unavailable, using HistGradientBoosting. Reason:', repr(e))
    from sklearn.ensemble import HistGradientBoostingClassifier
    booster = HistGradientBoostingClassifier(
        max_depth=4, learning_rate=0.05, max_iter=300,
        class_weight='balanced', random_state=42)
    model_name = 'HistGradientBoosting'

gb = Pipeline([('pre', pre), ('clf', booster)])
gb.fit(X_train, y_train)
gb_proba = evaluate(model_name, gb, X_test, y_test)

scale_pos_weight = 31.1
===== XGBoost =====
ROC-AUC : 0.589
PR-AUC  : 0.043  (base rate 0.031)
confusion matrix [tn fp / fn tp]:
[[11820  6801]
 [  311   288]]
              precision    recall  f1-score   support

           0      0.974     0.635     0.769     18621
           1      0.041     0.481     0.075       599

    accuracy                          0.630     19220
   macro avg      0.507     0.558     0.422     19220
weighted avg      0.945     0.630     0.747     19220



## 7. Feature importance

What (weakly) moves repeat. Read this as direction, not destiny, given the modest AUC.

In [10]:
feat_names = gb.named_steps['pre'].get_feature_names_out()
clf = gb.named_steps['clf']

if hasattr(clf, 'feature_importances_'):
    imp = pd.Series(clf.feature_importances_, index=feat_names).sort_values(ascending=False)
    print('Top 15 features:')
    print(imp.head(15).round(4))
else:
    print('Model exposes no feature_importances_; skipping.')

Top 15 features:
num__distinct_products                0.0448
num__review_score                     0.0388
num__item_count                       0.0384
num__total_freight                    0.0342
cat__main_payment_type_debit_card     0.0317
cat__customer_state_AL                0.0310
num__total_payment                    0.0306
cat__customer_state_DF                0.0303
cat__customer_state_CE                0.0289
cat__customer_state_RS                0.0277
cat__main_payment_type_credit_card    0.0266
num__delivery_time_days               0.0266
num__days_vs_estimate                 0.0263
cat__customer_state_SE                0.0262
cat__customer_state_RN                0.0261
dtype: float32


## 8. Honest read (fill in after running)

Write the real numbers here before they go anywhere near the resume:
- ROC-AUC (LR vs booster): ____
- PR-AUC vs base rate: does the model beat 0.031 by a meaningful margin, or barely?
- If AUC is ~0.6: state plainly that first-order experience weakly predicts repeat, and that the repeat
  problem is structural. That is a legitimate, defensible finding, not a failed model.
- Resume bullet: replace "XGBoost churn AUC 0.87" with the measured repeat-prediction number and the
  honest framing. Do not ship 0.87 on a model that scored 0.6.


## 9. Save model + per-customer scores

In [11]:
import joblib

# pick the better model by ROC-AUC for production scoring
lr_auc = roc_auc_score(y_test, lr_proba)
gb_auc = roc_auc_score(y_test, gb_proba)
best, best_name = (gb, model_name) if gb_auc >= lr_auc else (lr, 'LogisticRegression')
print(f"Saving best model by ROC-AUC: {best_name} ({max(lr_auc, gb_auc):.3f})")

joblib.dump(best, MODELS / 'repeat_model.pkl')

scored = first[['customer_unique_id', 'repeat']].copy()
scored['repeat_proba'] = best.predict_proba(X)[:, 1]
scored.to_parquet(PROC / 'customers_repeat_scored.parquet', index=False)

print('Saved:')
print(' - models/repeat_model.pkl')
print(' - data/processed/customers_repeat_scored.parquet', scored.shape)

Saving best model by ROC-AUC: XGBoost (0.589)
Saved:
 - models/repeat_model.pkl
 - data/processed/customers_repeat_scored.parquet (96096, 3)


## Done — hand to Day 9 (dashboard)

- `repeat_model.pkl` and `customers_repeat_scored.parquet` feed the dashboard's risk/propensity view.
- Note for the dashboard: a per-customer "at-risk" list is weak at a 3% base rate (almost everyone is
  "at risk" of not returning). The honest dashboard framing is **repeat-propensity distribution + the
  experience factors that move it**, not a ranked churn list. Use feature importance for the "what to fix" story.

**Day 9:** Streamlit dashboard. Executive summary, fulfillment funnel + delivery satisfaction, VoC
(themes/topics/emotion), segments + CLV, repeat-propensity. All inputs now exist in `data/processed/`.
